# Perch -> EfficientNet full FT

Stage 1 learns centered-crop Perch features with cosine loss. Stage 2 trains a new classifier with BCE and random focal crops. Both stages use the same saved fold. Start with `SMOKE_RUN=True`; then set it to `False` and Run All for the real experiment.


In [1]:
import json
import os

import torch

from birdclef.config import TrainConfig, get_config
from birdclef.models import AudioExperimentConfig, AudioModelConfig
from birdclef.models.distillation.config import DistillationConfig
from birdclef.runtime import apply_environment_defaults

apply_environment_defaults()
torch.set_num_threads(4)
defaults = get_config(verbose=False)

SMOKE_RUN = False
WANDB_OFFLINE = SMOKE_RUN
NAME = "distillation/EfficientNet-full-FT"
PREPARED_DIR = defaults.data.output_dir / "data_prep"
TEACHER_PATH = defaults.data.cache_dir / "models/perch_v2_backbone.onnx"
assert TEACHER_PATH.is_file(), TEACHER_PATH

loader = {
    "batch_size": 4 if SMOKE_RUN else 32,
    "val_batch_size": 16, "predict_batch_size": 16,
    "num_workers": 0,  # Avoid extra Python processes on Windows.
    "pin_memory": False, "seed": 42, "soundscape_probability": 0.1,
}
runtime = {"accelerator": "auto", "devices": 1}

def save_config(config, name):
    path = config.data.output_dir / "configs" / f"{config.run_name(name)}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(config.safe_dump(), indent=2), encoding="utf-8")
    if config.tracking.api_key is not None:
        os.environ["BCLEF_TRACKING__API_KEY"] = config.tracking.api_key.get_secret_value()
    return path


Detected environment: local
Loading default Config...


## Stage 1: features

One centered 5-second crop per focal clip; exact soundscape windows. Reads the existing cache only. Validation recordings are held out of training. Best checkpoint minimizes validation cosine loss.


In [2]:
feature_config = get_config(
    DistillationConfig,
    teacher_path=TEACHER_PATH,
    fold=0, smoke=SMOKE_RUN,
    model=AudioModelConfig(encoder_lr=3e-4),
    loader=loader, runtime=runtime,
    tracking={"offline": WANDB_OFFLINE},
    train=TrainConfig(
        max_epochs=1 if SMOKE_RUN else 5,
        early_stopping_patience=2,
        log_every_n_steps=1 if SMOKE_RUN else 10,
    ),
)
FEATURE_NAME = f"{NAME}/features"
FEATURE_DIR = feature_config.data.output_dir / feature_config.run_name(FEATURE_NAME)
feature_path = save_config(feature_config, FEATURE_NAME)


                                         DistillationConfig                                          
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Setting                       ┃ Value                                                             ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ app_name                      │ birdclef                                                          │
│ logging.directory             │ C:\_GitHub\birdclef-2026\logs                                     │
│ logging.level                 │ INFO                                                              │
│ logging.formatter             │ ColoredFormatter                                                  │
│ logging.date_format           │ %d-%m-%Y %H:%M:%S                                                 │
│ logging.full_color            │ True                                                              │
│ logging.include_function      │ True                                                              │
│ logging.success_level         │ 69                                                                │
├───────────────────────────────┼───────────────────────────────────────────────────────────────────┤
│ data.competition_dir          │ C:\_GitHub\birdclef-2026\data\competition                         │
│ data.cache_dir                │ C:\_GitHub\birdclef-2026\data\cache                               │
│ data.output_dir               │ C:\_GitHub\birdclef-2026\outputs                                  │
├───────────────────────────────┼───────────────────────────────────────────────────────────────────┤
│ split.validation_fraction     │ 0.2                                                               │
│ split.seed                    │ 42                                                                │
│ split.group_column            │ None                                                              │
│ split.stratify                │ False                                                             │
├───────────────────────────────┼───────────────────────────────────────────────────────────────────┤
│ tracking.project              │ birdclef-2026                                                     │
│ tracking.entity               │ dmykhailov-kyiv-school-of-economics                               │
│ tracking.offline              │ False                                                             │
│ tracking.api_key              │ **********                                                        │
├───────────────────────────────┼───────────────────────────────────────────────────────────────────┤
│ loader.batch_size             │ 32                                                                │
│ loader.val_batch_size         │ 16                                                                │
│ loader.predict_batch_size     │ 16                                                                │
│ loader.num_workers            │ 0                                                                 │
│ loader.pin_memory             │ False                                                             │
│ loader.seed                   │ 42                                                                │
│ loader.soundscape_probability │ 0.1                                                               │
├───────────────────────────────┼───────────────────────────────────────────────────────────────────┤
│ lr_scheduler.name             │ none                                                              │
│ lr_scheduler.t_max            │ 10                                                                │
│ lr_scheduler.min_lr           │ 0.0                                                               │
│ lr_scheduler.factor           │ 0.1                                                               │
│ lr_scheduler.patienc

In [ ]:
%run -m birdclef.models.distillation.fit -- --config "{feature_path}" --name "{FEATURE_NAME}" --prepared-dir "{PREPARED_DIR}"


W1029 20:32:07.872000 8956 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Detected environment: local


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: Appending key for api.forge.coreweave.com to your netrc file: C:\Users\dennm\_netrc
wandb: Currently logged in as: dmykhailov (dmykhailov-kyiv-school-of-economics) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


Seed set to 42


Output()

...

Trainable params: 6.0 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 6.0 M                                                                                                
Total estimated model params size (MB): 23.898                                                                     
Modules in train mode: 334                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

c:\_GitHub\birdclef-2026\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\_GitHub\birdclef-2026\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.
c:\_GitHub\birdclef-2026\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

## Stage 2: supervised full fine-tuning

Load only `encoder.pt`; discard the projection and initialize a fresh linear head for 234 classes. Both encoder and head use LR `3e-4` (D1). Normal random focal crops resume. Select the best checkpoint by validation ROC-AUC.


In [ ]:
classifier_config = get_config(
    AudioExperimentConfig,
    fold=feature_config.fold, smoke=SMOKE_RUN,
    model=AudioModelConfig(
        encoder_weights=FEATURE_DIR / "encoder.pt",
        encoder_lr=3e-4, head_lr=3e-4,
        unfreeze_epoch=0, unfreeze_last_n=None,
    ),
    loader=loader, runtime=runtime,
    tracking={"offline": WANDB_OFFLINE},
    train=TrainConfig(
        max_epochs=1 if SMOKE_RUN else 5,
        checkpoint_metric="val/auc", checkpoint_mode="max",
        early_stopping_patience=2,
        log_every_n_steps=1 if SMOKE_RUN else 10,
    ),
)
CLASSIFIER_NAME = f"{NAME}/full-ft"
RUN_DIR = classifier_config.data.output_dir / classifier_config.run_name(CLASSIFIER_NAME)
classifier_path = save_config(classifier_config, CLASSIFIER_NAME)


                                               AudioExperimentConfig                                               
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Setting                       ┃ Value                                                                           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ app_name                      │ birdclef                                                                        │
│ logging.directory             │ C:\_GitHub\birdclef-2026\logs                                                   │
│ logging.level                 │ INFO                                                                            │
│ logging.formatter             │ ColoredFormatter                                                                │
│ logging.date_format           │ %d-%m-%Y %H:%M:%S                                                               │
│ logging.full_color            │ True                                                                            │
│ logging.include_function      │ True                                                                            │
│ logging.success_level         │ 69                                                                              │
├───────────────────────────────┼─────────────────────────────────────────────────────────────────────────────────┤
│ data.competition_dir          │ C:\_GitHub\birdclef-2026\data\competition                                       │
│ data.cache_dir                │ C:\_GitHub\birdclef-2026\data\cache                                             │
│ data.output_dir               │ C:\_GitHub\birdclef-2026\outputs                                                │
├───────────────────────────────┼─────────────────────────────────────────────────────────────────────────────────┤
│ split.validation_fraction     │ 0.2                                                                             │
│ split.seed                    │ 42                                                                              │
│ split.group_column            │ None                                                                            │
│ split.stratify                │ False                                                                           │
├───────────────────────────────┼─────────────────────────────────────────────────────────────────────────────────┤
│ tracking.project              │ birdclef-2026                                                                   │
│ tracking.entity               │ dmykhailov-kyiv-school-of-economics                                             │
│ tracking.offline              │ False                                                                           │
│ tracking.api_key              │ **********                                                                      │
├───────────────────────────────┼─────────────────────────────────────────────────────────────────────────────────┤
│ loader.batch_size             │ 32                                                                              │
│ loader.val_batch_size         │ 16                                                                              │
│ loader.predict_batch_size     │ 16                                                                              │
│ loader.num_workers            │ 0                                                                               │
│ loader.pin_memory             │ False                                                                           │
│ loader.seed                   │ 42                                                                              │
│ loader.soundscape_probability │ 0.1                                                                             │
├───────────────────────────────┼───────────────────────

In [6]:
classifier_config.train.max_epochs = 3
classifier_path = save_config(classifier_config, CLASSIFIER_NAME)

In [7]:
%run -m birdclef.models.fit -- --config "{classifier_path}" --name "{CLASSIFIER_NAME}" --prepared-dir "{PREPARED_DIR}"

29-10-2026 22:27:32  INFO      [models/fit:66]  Run: distillation/EfficientNet-full-FT/full-ft; training examples: 35769; validation windows: 149


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.forge.coreweave.com.
wandb: Appending key for api.forge.coreweave.com to your netrc file: C:\Users\dennm\_netrc


Seed set to 42
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\_GitHub\birdclef-2026\.venv\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:881: Checkpoint directory C:\_GitHub\birdclef-2026\outputs\distillation\EfficientNet-full-FT\full-ft\checkpoints exists and is not empty.
Restoring states from the checkpoint path at C:\_GitHub\birdclef-2026\outputs\distillation\EfficientNet-full-FT\full-ft\checkpoints\last.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\_GitHub\birdclef-2026\.venv\Lib\site-packages\lightning\pytorch\utilities\model_summary\model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ criterion │ BCEWithLogitsLoss   │      0 │ train │     0 │
│ 1 │ encoder   │ EfficientNetEncoder │  4.0 M │ train │     0 │
│ 2 │ head      │ Sequential          │  299 K │ train │     0 │
└───┴───────────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 4.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.3 M                                                                                                
Total estimated model params size (MB): 17.227                                                                     
Modules in train mode: 336                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Restored all states from the checkpoint at C:\_GitHub\birdclef-2026\outputs\distillation\EfficientNet-full-FT\full-ft\checkpoints\last.ckpt
c:\_GitHub\birdclef-2026\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\_GitHub\birdclef-2026\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.
`Trainer.fit` stopped: `max_epochs=3` reached.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and

Predicting: |          | 0/? [00:00<?, ?it/s]

best/val_auc,▁
perf/fit_seconds,▁
trainer/global_step,▁▁▁
best/val_auc,0.89463
epoch,3
lr-AdamW/encoder,0.0003
lr-AdamW/head,0.0003
perf/fit_seconds,0.24407
train/loss_epoch,0.01533
train/loss_step,0.01522
trainer/global_step,0


Saved training run: C:\_GitHub\birdclef-2026\outputs\distillation\EfficientNet-full-FT\full-ft


In [8]:
import pandas as pd
from IPython.display import display

display(pd.read_json(RUN_DIR / "validation-metrics.json", typ="series"))
print(f"Classifier for export: {RUN_DIR}")


auc                    0.894632
auc_classes                  28
windows                     149
protocol       file-gkf5/fold-0
dtype: object

Classifier for export: C:\_GitHub\birdclef-2026\outputs\distillation\EfficientNet-full-FT\full-ft


## Export

Upload this ZIP to your models Dataset. Update the wheelhouse Dataset with the current package too (`python -m scripts.kaggle.wheelhouse publish`). The submission notebook stays unchanged; it loads a regular EfficientNet checkpoint, without Perch, cached features or the Stage-1 projection. Select the exported student with `MODEL_WEIGHTS = {"full-ft": 1.0}`.


In [ ]:
%run -m birdclef.submission.export -- --run "{RUN_DIR}" --all --output build/perch-distilled-efficientnet.zip